# Interpretation experiment for 2D Reaction-Diffusion

This notebook mirrors the 1D interpretation workflow, but uses 2D reaction-diffusion data and heatmaps over the x/y grid. It creates the small mini-figures for the interpretation results for 2d reaction-diffusion

In [ ]:
import json
import os
import sys
from pathlib import Path

import importlib

import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)

from src.train import build_configs
from src.data.equation_datamodule import EquationModule
from src.models.base_lightning_model import PDELightningModule

import src.utils.evaluation.evaluation as ev
import src.utils.visualisations.visualise_results as vis

importlib.reload(ev)
importlib.reload(vis)

EQUATION = "reactiondiffusion2d"
RUN_DIR = Path("outputs/reactiondiffusion2d_benchmark/reactiondiffusion2d_benchmark_late_fusion_sparsity_sweep_sparsity_weight-1e-4_seed234")

resolved = json.loads((RUN_DIR / "resolved_config.json").read_text(encoding="utf-8"))
equation = resolved["equation"]
model_name = resolved["model"]
training = resolved.get("training", {})

resolved_model_name, model_config, datamodule_config = build_configs(equation, model_name)
model_config.update(resolved.get("model_config", {}))
datamodule_config.update(resolved.get("datamodule_config", {}))

dm = EquationModule(**datamodule_config)
dm.setup(stage=None)
dm.batch_size_test = 500
test_id_loader = dm.test_dataloader()

ckpt_dir = RUN_DIR / "checkpoints"
ckpt_path = ckpt_dir / "last.ckpt"
if not ckpt_path.exists():
    best_ckpts = sorted(ckpt_dir.glob("epoch=*.ckpt"))
    if not best_ckpts:
        raise FileNotFoundError(f"No checkpoint found in: {ckpt_dir}")
    ckpt_path = best_ckpts[-1]

model = PDELightningModule.load_from_checkpoint(
    str(ckpt_path),
    model_name=resolved_model_name,
    model_config=model_config,
    learning_rate=float(training.get("learning_rate", 1e-3)),
    log_hyperparameters=False,
    strict=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
print("Using run_dir:", RUN_DIR)
print("Checkpoint:", ckpt_path)

In [ ]:
batch = next(iter(test_id_loader))
batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

state = batch["state"]  # expected: [B, T, X, Y, V]
B, T, X, Y, V = state.shape

grid = batch["x"]  # expected: [B, X, Y, 2]
x_coord = grid[0, :, 0, 0].detach().cpu().numpy().reshape(-1)
y_coord = grid[0, 0, :, 1].detach().cpu().numpy().reshape(-1)

t = batch.get("t", None)
if t is not None:
    t_cpu = t.detach().cpu()
    if t_cpu.ndim == 1:
        t_vec = t_cpu
    else:
        t_vec = t_cpu[0].reshape(-1)
    dt = float((t_vec[1] - t_vec[0]).item()) if t_vec.numel() > 1 else np.nan
else:
    t_vec = torch.arange(T, dtype=torch.float32)
    dt = 1.0

print(f"Batch shape [B,T,X,Y,V]: {tuple(state.shape)}")
print(f"Grid shape [B,X,Y,2]: {tuple(grid.shape)}")
print(f"Estimated time step dt: {dt}")

In [ ]:
FIGSIZE = (0.9, 0.9)
FONTSIZE = 7
PLOT_RECT = [0.12, 0.14, 0.68, 0.73]
CBAR_RECT = [0.83, 0.14, 0.03, 0.73]
plt.rcParams.update({"font.size": FONTSIZE})

def _new_heatmap_axes():
    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_axes(PLOT_RECT)
    cax = fig.add_axes(CBAR_RECT)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.tick_params(labelsize=FONTSIZE, pad=0.5)
    cax.tick_params(labelsize=FONTSIZE, pad=0.5)
    return fig, ax, cax

def _save_heatmap(field, out_path, cbar_label=None, cmap="viridis", vmin=None, vmax=None):
    fig, ax, cax = _new_heatmap_axes()
    im = ax.imshow(
        field.T,
        origin="lower",
        aspect="auto",
        extent=[x_coord[0], x_coord[-1], y_coord[0], y_coord[-1]],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    fig.colorbar(im, cax=cax)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, format="svg", bbox_inches="tight", pad_inches=0.01)
    plt.show()
    plt.close(fig)
    print(f"Saved: {out_path}")

In [ ]:
sample_idx = 1
u0 = state[sample_idx, 0].detach().cpu().numpy()  # [X, Y, V]

for var_idx in range(u0.shape[-1]):
    out_path = Path(f"outputs/interpretation_2d/reactiondiffusion2d/initial_var{var_idx}.svg")
    _save_heatmap(
        u0[:, :, var_idx],
        out_path,
        cbar_label=None,
    )

u = u0[:, :, 0]
dx = float(x_coord[1] - x_coord[0])
dy = float(y_coord[1] - y_coord[0])

du_dx = np.gradient(u, dx, axis=0)
du2_dx2 = np.gradient(du_dx, dx, axis=0)
du_dy = np.gradient(u, dy, axis=1)
du2_dy2 = np.gradient(du_dy, dy, axis=1)

v = u0[:, :, 1]
dv_dx = np.gradient(v, dx, axis=0)
dv2_dx2 = np.gradient(dv_dx, dx, axis=0)
dv_dy = np.gradient(v, dy, axis=1)
dv2_dy2 = np.gradient(dv_dy, dy, axis=1)
laplacian = np.gradient(du_dx, dx, axis=0) + np.gradient(du_dy, dy, axis=1)

Du = 1e-3
Dv = 5e-3

params = batch["parameter"][sample_idx].detach().cpu().reshape(-1)[0].item()

u_np = Du * du2_dx2 + Du * du2_dy2 + u - u**3 - v
v_np = Dv * dv2_dx2 + Dv * dv2_dy2 + u - v
u_p = -params*u**0
out_path = Path("outputs/interpretation_2d/reactiondiffusion2d/u_np.svg")
_save_heatmap(
    u_np,
    out_path,
    cbar_label=None,
)
out_path = Path("outputs/interpretation_2d/reactiondiffusion2d/u_p.svg")
_save_heatmap(
    u_p,
    out_path,
    cbar_label=None,
    vmin = -0.005,
    vmax = -0.003
)
out_path = Path("outputs/interpretation_2d/reactiondiffusion2d/v_np.svg")
_save_heatmap(
    v_np,
    out_path,
    cbar_label=None,
    vmin=-0.22,
    vmax=0.22
)

In [ ]:
core = model.model
assert hasattr(core, "fno"), "Loaded model wrapper does not expose .fno"
assert hasattr(core, "library"), "Loaded model wrapper does not expose .library"
assert hasattr(core, "regression"), "Loaded model wrapper does not expose .regression"

with torch.no_grad():
    input_state_t0 = state[:, 0, :, :, :]  # [B, X, Y, V]
    h = core.fno(input_state_t0, batch["x"])  # typically [B, X, Y, 1, H]
    print(h.shape)
    if h.ndim == 5:
        h = h.squeeze(-2)  # -> [B, X, Y, H]
    hidden_first = h[sample_idx].detach().cpu().numpy()  # [X, Y, H]

print("Hidden state shape for first trajectory at t0:", hidden_first.shape)

for ch_idx in range(hidden_first.shape[-1]):
    out_path = Path(f"outputs/interpretation_2d/reactiondiffusion2d/hiddenstate_{ch_idx}.svg")
    _save_heatmap(
        hidden_first[:, :, ch_idx],
        out_path,
        cbar_label=None,
    )

In [ ]:
W = core.regression.get_coefficients().detach().cpu()  # [output_dim, library_size]
print("Coefficient matrix shape:", tuple(W.shape))
print("Coefficient matrix:\n", W.numpy())

hidden_dim = hidden_first.shape[-1]
param_dim = batch["parameter"].shape[-1]
terms = core.library.get_feature_names(hidden_dim=hidden_dim, param_dim=param_dim)

if len(terms) != W.shape[1]:
    print(f"Warning: number of terms ({len(terms)}) != number of coefficients ({W.shape[1]})")

coef_table = pd.DataFrame({"term": terms[:W.shape[1]]})
for out_i in range(W.shape[0]):
    coef_table[f"coef_out{out_i}"] = W[out_i, :len(coef_table)].numpy()

print("\nTerm -> coefficients:")
display(coef_table)

topk = min(20, len(coef_table))
top_terms = coef_table.assign(abs_coef=np.abs(coef_table["coef_out0"])).sort_values("abs_coef", ascending=False).head(topk)
print(f"\nTop {topk} terms by |coef_out0|:")
display(top_terms[["term", "coef_out0", "abs_coef"]])

In [ ]:
params = batch["parameter"][sample_idx].detach().cpu().reshape(-1)[0].item()
coef_0 = W[0, 0].item()

hidden_scaled = hidden_first[:, :, 0] / max(dt, 1e-8) * params * coef_0
out_path = Path("outputs/interpretation_2d/reactiondiffusion2d/hiddenstate_0_scaled.svg")
_save_heatmap(
    hidden_scaled,
    out_path,
    cbar_label=r"$(h_0/\Delta t)\cdot \mathrm{param}\cdot c_0$",
)

pred_next = model(input_state_t0, batch["parameter"], batch["x"])  # [B, X, Y, V]
print("Model output shape:", pred_next.shape)

for var_idx in range(pred_next.shape[-1]):
    out_path = Path(f"outputs/interpretation_2d/reactiondiffusion2d/output_var{var_idx}.svg")
    _save_heatmap(
        pred_next[sample_idx, :, :, var_idx].detach().cpu().numpy(),
        out_path,
        cbar_label=f"pred u{var_idx}",
    )

In [ ]:
# Decompose predictions into individual library term contributions
# Identify which hidden states are available
hidden_dim = hidden_first.shape[-1]
available_hidden_terms = [f"u{i+1}" for i in range(hidden_dim)]

# Build term groups
# Group 1: Constant + hidden states only [1, u1, u2, u3, ...]
# Group 2: Parameter and parameter-weighted terms [p1, u1*p1, u2*p1, u3*p1, ...]
group1_terms = ["1"] + available_hidden_terms
group2_terms = ["p1"] + [f"{h}*p1" for h in available_hidden_terms]

# Find indices for all terms
term_to_idx = {t: i for i, t in enumerate(terms)}
print("Available terms in library:", terms)
print("\nGroup 1 terms (state-only):", group1_terms)
print("Group 2 terms (parameter-weighted):", group2_terms)

output_names = ["u", "v"]  # Variable names for outputs

# For each output variable, create heatmaps for each group
for out_idx in range(W.shape[0]):
    var_name = output_names[out_idx] if out_idx < len(output_names) else f"var{out_idx}"
    print(f"\n=== Creating decomposition plots for output {out_idx} ({var_name}) ===")
    
    # Group 1: Sum of [1 + u1 + u2 + u3 + ...] weighted by coefficients, divided by dt
    group1_contribution = np.zeros_like(hidden_first[:, :, 0])
    for term_name in group1_terms:
        if term_name in term_to_idx:
            term_idx = term_to_idx[term_name]
            coeff = W[out_idx, term_idx].item()
            
            if term_name == "1":
                term_value = np.ones_like(hidden_first[:, :, 0])
            else:
                # Extract hidden state index from "u1", "u2", etc.
                h_idx = int(term_name[1]) - 1
                if h_idx < hidden_first.shape[-1]:
                    term_value = hidden_first[:, :, h_idx]
                else:
                    continue
            
            group1_contribution += coeff * term_value / max(dt, 1e-8)
            print(f"  Added term '{term_name}' (coeff={coeff:.6f})")
    
    if out_idx == 0:
        vmin, vmax = None, None
    else:
        vmin, vmax = -0.22, 0.22
    out_path = Path(f"outputs/interpretation_2d/reactiondiffusion2d/{var_name}_group1_stateonly.svg")
    _save_heatmap(
        group1_contribution,
        out_path,
        cbar_label=None,
        vmin=vmin,
        vmax=vmax
    )
    print(f"  Saved Group 1 (state-only) to {out_path.name}")
    
    # Group 2: Sum of [p1 + u1*p1 + u2*p1 + u3*p1 + ...] weighted by coefficients, divided by dt
    group2_contribution = np.zeros_like(hidden_first[:, :, 0])
    for term_name in group2_terms:
        if term_name in term_to_idx:
            term_idx = term_to_idx[term_name]
            coeff = W[out_idx, term_idx].item()
            
            if term_name == "p1":
                # This term is the parameter itself
                term_value = np.ones_like(hidden_first[:, :, 0]) * params
            else:
                # Extract hidden state from terms like "u1*p1"
                h_name = term_name.split("*")[0]
                h_idx = int(h_name[1]) - 1
                if h_idx < hidden_first.shape[-1]:
                    term_value = hidden_first[:, :, h_idx] * params
                else:
                    continue
            
            group2_contribution += coeff * term_value / max(dt, 1e-8)
            print(f"  Added term '{term_name}' (coeff={coeff:.6f})")
    
    if out_idx == 0:
        vmin, vmax = -0.005, -0.003
    else:
        vmin, vmax = -0.001, 0.001
    out_path = Path(f"outputs/interpretation_2d/reactiondiffusion2d/{var_name}_group2_paramweighted.svg")
    _save_heatmap(
        group2_contribution,
        out_path,
        cbar_label=None,
        vmin = vmin,
        vmax = vmax
    )
    print(f"  Saved Group 2 (parameter-weighted) to {out_path.name}")

print("\n✓ All decomposition plots created!")